# Notebook 06 (Exp 2) — BLEU + BERTScore Evaluation

Evaluates exp2 models on the held-out test set.

New vs exp1:
- Loads models from `outputs/exp2/` (not exp1)
- Adds **BERTScore** (F1, precision, recall) — semantic similarity metric more appropriate for style transfer than BLEU
- Saves scores to `outputs/exp2/results/bleu_scores.json`

In [ ]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import AutoPeftModelForCausalLM

sys.path.insert(0, str(Path('..').resolve()))
from src.evaluation import compute_bleu_scores, compute_bert_scores, run_batch_inference

ROOT          = Path('..').resolve()
PROCESSED_DIR = ROOT / 'data' / 'processed'
LORA_PATH     = ROOT / 'outputs' / 'exp2' / 'lora'  / 'final_adapter'
FFT_PATH      = ROOT / 'outputs' / 'exp2' / 'fft'   / 'final_model'
RESULTS_DIR   = ROOT / 'outputs' / 'exp2' / 'results'
FIG_DIR       = ROOT / 'outputs' / 'exp2' / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

TEST_JSONL = PROCESSED_DIR / 'test.jsonl'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style='whitegrid')
print('Setup complete.')


## 1. Load LoRA Model and Run Inference

In [ ]:
print('Loading Exp 2 LoRA model ...')
model_lora = AutoPeftModelForCausalLM.from_pretrained(
    str(LORA_PATH),
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model_lora.eval()
tok_lora = AutoTokenizer.from_pretrained(str(LORA_PATH))
print('LoRA model loaded.')

hyps_lora, refs_lora = run_batch_inference(
    model_lora, tok_lora,
    test_jsonl_path=TEST_JSONL,
    direction='mod2shak',
    max_new_tokens=256,
)
print(f'\nInference complete: {len(hyps_lora)} test examples')


In [ ]:
import gc
del model_lora
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('LoRA model unloaded.')


## 2. Load FFT Model and Run Inference

In [ ]:
print('Loading Exp 2 FFT model ...')
model_fft = AutoModelForCausalLM.from_pretrained(
    str(FFT_PATH),
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
model_fft.eval()
tok_fft = AutoTokenizer.from_pretrained(str(FFT_PATH))
print('FFT model loaded.')

hyps_fft, refs_fft = run_batch_inference(
    model_fft, tok_fft,
    test_jsonl_path=TEST_JSONL,
    direction='mod2shak',
    max_new_tokens=256,
)
print(f'\nInference complete: {len(hyps_fft)} test examples')


In [ ]:
del model_fft
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('FFT model unloaded.')


## 3. Compute BLEU and ChrF Scores

In [ ]:
scores_lora, sent_bleus_lora = compute_bleu_scores(hyps_lora, refs_lora)
scores_fft,  sent_bleus_fft  = compute_bleu_scores(hyps_fft,  refs_fft)

print('=== Exp 2 LoRA Scores ===')
for k, v in scores_lora.items():
    print(f'  {k:<25} {v}')

print('\n=== Exp 2 FFT Scores ===')
for k, v in scores_fft.items():
    print(f'  {k:<25} {v}')


## 4. BERTScore — Semantic Similarity

BERTScore captures semantic equivalence rather than n-gram overlap. A valid Shakespearean paraphrase that uses different vocabulary still scores high. Uses `distilbert-base-uncased` for speed; switch to `roberta-large` for publication-quality scores.

In [ ]:
print('Computing BERTScore (LoRA) ...')
bert_lora = compute_bert_scores(hyps_lora, refs_lora, model_type='distilbert-base-uncased')
print('LoRA BERTScore:', bert_lora)

print('\nComputing BERTScore (FFT) ...')
bert_fft  = compute_bert_scores(hyps_fft,  refs_fft,  model_type='distilbert-base-uncased')
print('FFT BERTScore: ', bert_fft)


## 5. Save All Scores

In [ ]:
all_scores = {
    'lora': {**scores_lora, **bert_lora},
    'fft':  {**scores_fft,  **bert_fft},
}
with open(RESULTS_DIR / 'bleu_scores.json', 'w') as f:
    json.dump(all_scores, f, indent=2)
print('Scores saved to outputs/exp2/results/bleu_scores.json')
print(json.dumps(all_scores, indent=2))


## 6. Sentence BLEU Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, label, color in zip(
    axes,
    [sent_bleus_lora, sent_bleus_fft],
    ['LoRA (Exp 2, Qwen2.5-3B)', 'FFT (Exp 2, Qwen2.5-1.5B)'],
    ['steelblue', 'darkorange']
):
    ax.hist(scores, bins=30, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(np.mean(scores), color='red', linestyle='--',
               label=f'Mean = {np.mean(scores):.1f}')
    ax.axvline(np.median(scores), color='green', linestyle=':',
               label=f'Median = {np.median(scores):.1f}')
    ax.set_title(f'Sentence BLEU Distribution — {label}')
    ax.set_xlabel('Sentence BLEU Score')
    ax.set_ylabel('Count')
    ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / 'exp2_bleu_distributions.png', dpi=150)
plt.show()


## 7. Sample Qualitative Comparisons

In [ ]:
with open(TEST_JSONL, encoding='utf-8') as f:
    test_records = [json.loads(line) for line in f if line.strip()]

N = 8
print(f'{"Modern English Source":<45} {"LoRA Output":<45} {"FFT Output":<45} {"Reference":<45}')
print('\u2500' * 185)

for i in range(N):
    msgs = test_records[i]['messages']
    src  = next(m['content'] for m in msgs if m['role'] == 'user')
    ref  = next(m['content'] for m in msgs if m['role'] == 'assistant')

    def trunc(s, n=42):
        return s[:n] + '...' if len(s) > n else s

    print(f'{trunc(src):<45} {trunc(hyps_lora[i]):<45} {trunc(hyps_fft[i]):<45} {trunc(ref):<45}')
